# 행렬곱에서 `nn.Linear`까지: 저장 파라미터, 순서, 안정성

- 기준 TIL: [2026-08-18](../../til/2026/08/2026-08-18.md)
- 관련 강의자료: [1장 3강](../../materials/private/kant-basic-math/01-03_행렬_연산과_딥러닝_레이어.md), [1장 4강](../../materials/private/kant-basic-math/01-04_특수_행렬과_행렬_연산_성질.md)
- 강의 제공 실습: [01-01](../../materials/private/kant-basic-math/course-provided-practice/01-01_벡터_노름_정규화.md), [01-02](../../materials/private/kant-basic-math/course-provided-practice/01-02_내적과_코사인_유사도.md), [01-03](../../materials/private/kant-basic-math/course-provided-practice/01-03_행렬곱과_완전연결층.md), [01-04](../../materials/private/kant-basic-math/course-provided-practice/01-04_특수_행렬과_행렬_연산_성질.md)
- 앞선 실행 기록: [벡터 정규화](vector-normalization-axis.ipynb), [내적과 코사인 유사도](dot-product-cosine-ranking.ipynb)
- 난이도: Core
- 상태: 완료


## 강의 제공 실습을 어떻게 다듬었는가

- `01-01`의 벡터 크기·방향 구분과 `01-02`의 여러 내적을 한 번에 계산하는 관점은 선수 연결로 유지했다. 두 주제는 위의 기존 실행 기록에서 이미 다뤘으므로 같은 문제를 반복하지 않는다.
- `01-03`의 실행 전 Shape 예측과 `X @ W + b`를 핵심 뼈대로 유지하고, PyTorch가 저장하는 `weight`의 배치 방향을 직접 맞추는 과제를 보충했다.
- `01-04`의 대각행렬·비교환성과 행렬식·조건수 비교를 작은 변경 예제로 유지했다.
- 이번 목표는 데이터 처리나 지표 해석이 아니라 행렬 계산 메커니즘이므로 UCI 데이터 다운로드, pandas 전처리, 완성 답안과 예시 출력은 가져오지 않았다.


## 왜 지금 이 실습을 하는가

- TIL과 학습 답변에서 확인된 이해: 행렬곱의 한 칸이 행과 열의 내적이라는 점, `nn.Linear(in, out)`이 `(out, in)` Shape의 `weight`를 저장한다는 점, 같은 레이어가 같은 파라미터를 재사용한다는 점을 설명했다. 대각행렬의 축별 작용, 행렬곱의 일반적인 비교환성, 전치 시 순서 반전, 행렬식과 조건수의 역할도 구분했다.
- 앞선 실행 기록에서 확인된 이해: 마지막 축 정규화와 broadcasting을 실행 결과로 해석했고, 벡터 크기 변화가 내적과 코사인 유사도에 서로 다르게 작용함을 비교했다.
- 이번에 확인할 부족한 부분: 직접 정한 숫자로 `nn.Linear`의 저장 파라미터를 바꾸고, 명시적인 `X @ W + b`와 출력이 같음을 실행으로 검증한 기록은 없다. 특수 행렬의 곱 순서와 `det`·조건수 차이도 바뀐 숫자에서 코드로 확인할 필요가 있다.
- 핵심 질문: `nn.Linear`가 저장한 `weight`와 `bias`는 명시적인 행렬곱과 어떻게 같은 출력을 만들며, 가중치 구조와 곱 순서는 출력과 수치적 안정성에 어떤 차이를 만드는가?


## 선수개념과 완료 기준

선수개념은 벡터의 내적, 행렬곱 Shape 규칙, 전치, broadcasting, 단위·대각·대칭행렬, 행렬식과 조건수의 역할이다.

- [x] 실행 전에 모든 주요 Tensor의 Shape과 첫 출력값 하나를 예측했다.
- [x] 한 행의 내적 계산과 배치 행렬곱 결과를 연결했다.
- [x] 직접 정한 `W`, `b`를 `nn.Linear`에 넣고 명시적인 계산과 같은지 확인했다.
- [x] `bias`가 배치의 각 행에 더해지는 과정을 명시적인 Tensor로 확인했다.
- [x] 대각 스케일과 대칭 혼합의 순서를 바꾼 결과를 비교하고 원인을 설명했다.
- [x] 행렬식 크기만으로 수치 안정성을 판단할 수 없는 예를 실행 결과로 해석했다.
- [x] 관찰한 결과가 보여주는 범위와 한계 하나를 적었다.


## 실행 전 예상

준비 셀의 숫자를 읽은 뒤, 이후 계산 셀을 실행하기 전에 먼저 작성한다.

1. `X`, `W_io`, `b`, `Y_manual`의 Shape은 각각 무엇일까?
    - (2, 3), (3, 2), (2,), (2, 2)
2. `Y_manual[0, 0]`은 `X`의 어느 행과 `W_io`의 어느 열을 사용하며, 손계산 값은 얼마일까?
    - X의 1행과 W_io의 1열, 2.7
3. 같은 계산을 하는 `nn.Linear`의 `weight.shape`은 무엇이며 `W_io`를 그대로 복사할 수 있을까?
    - (2, 3), 전치한 뒤 복사해야한다.
4. `b`를 배치 크기만큼 명시적으로 늘리면 어떤 Shape과 값 배치가 될까?
    - bias.shape
5. 대각 스케일과 성분 혼합의 순서를 바꾸면 결과가 같을까?
    - 결과가 달라진다.
6. `det`가 더 작은 행렬과 조건수가 더 큰 행렬은 항상 같은 행렬일까?
    - 아니다.


In [2]:
# 준비: 위 예상부터 작성한 뒤 실행하세요.
import numpy as np
import torch
from torch import nn

torch.manual_seed(42)
torch.set_default_dtype(torch.float64)

# W_io는 수식 X @ W + b에서 쓰는 (입력, 출력) 배치입니다.
X = torch.tensor([
    [1.0, 2.0, -1.0],
    [0.5, -2.0, 3.0],
])
W_io = torch.tensor([
    [0.2, -0.5],
    [1.0, 0.3],
    [-0.4, 0.8],
])
b = torch.tensor([0.1, -0.2])

print('X.shape:', tuple(X.shape))
print('W_io.shape:', tuple(W_io.shape))
print('b.shape:', tuple(b.shape))


X.shape: (2, 3)
W_io.shape: (3, 2)
b.shape: (2,)


## 1. 한 번의 내적에서 배치 행렬곱으로

1. `Y_manual[0, 0]`을 이루는 곱셈과 덧셈을 종이에 먼저 적고 `first_output_0`으로 계산한다.
2. `Y_manual = X @ W_io + b`에 해당하는 코드를 직접 완성한다.
3. 손계산한 값과 `Y_manual[0, 0]`이 같은지 확인한다.
4. 결과의 각 행과 각 열이 무엇을 뜻하는지 적는다.


In [4]:
# TODO 1-A: 첫 번째 샘플과 첫 번째 출력 열의 내적에 bias를 더하세요.
first_output_0 = 2.7

# TODO 1-B: 전체 배치를 한 번에 계산하세요.
Y_manual = X @ W_io + b

assert first_output_0 is not None, '먼저 한 칸을 직접 계산하세요.'
assert Y_manual is not None, '배치 행렬곱을 완성하세요.'

print('first_output_0:', first_output_0)
print('Y_manual.shape:', tuple(Y_manual.shape))
print('한 칸과 배치 결과 일치:', torch.allclose(torch.as_tensor(first_output_0), Y_manual[0, 0]))
print('Y_manual:\n', Y_manual)


first_output_0: 2.7
Y_manual.shape: (2, 2)
한 칸과 배치 결과 일치: True
Y_manual:
 tensor([[ 2.7000, -0.9000],
        [-3.0000,  1.3500]])


## 2. 같은 숫자를 `nn.Linear`의 저장 파라미터에 넣기

1. `X`의 마지막 축과 원하는 출력 축을 보고 `nn.Linear`를 만든다.
2. `layer.weight`와 `W_io`의 Shape을 비교해 저장 배치에 맞게 값을 복사한다. `bias`도 같은 방법으로 복사한다.
3. 복사 전후 `weight` 객체의 `id`를 비교해 객체가 바뀐 것인지, 객체 안의 숫자가 바뀐 것인지 확인한다.
4. `layer(X)`와 `Y_manual`이 같은지 확인한다.
5. `b`를 `Y_manual`과 같은 Shape으로 명시적으로 늘리고, broadcasting 계산과 같은지 확인한다.
6. 출력된 `named_parameters()`의 이름과 Shape이 자동 등록에 관해 무엇을 보여주는지 적는다.


In [5]:
# TODO 2-A: 입력 차원과 출력 차원에 맞는 레이어를 만드세요.
layer = nn.Linear(3, 2)
assert layer is not None, 'nn.Linear 레이어를 먼저 만드세요.'

weight_id_before = id(layer.weight)

with torch.no_grad():
    # TODO 2-B: W_io와 b를 PyTorch의 저장 배치에 맞게 복사하세요.
    layer.weight.copy_(W_io.T)
    layer.bias.copy_(b)

weight_id_after = id(layer.weight)
Y_layer = layer(X)

# TODO 2-C: b를 Y_manual과 같은 Shape으로 명시적으로 늘리세요.
b_expanded = b.unsqueeze(0).expand_as(Y_manual)
assert b_expanded is not None, 'broadcasting을 눈으로 확인할 Tensor를 만드세요.'

print('저장된 weight.shape:', tuple(layer.weight.shape))
print('weight 객체 id 유지:', weight_id_before == weight_id_after)
print('등록된 파라미터:', [(name, tuple(param.shape)) for name, param in layer.named_parameters()])
print('수동 계산과 layer 출력 일치:', torch.allclose(Y_manual, Y_layer))
print('b_expanded.shape:', tuple(b_expanded.shape))
print('명시적 bias와 broadcasting 일치:', torch.allclose(X @ W_io + b_expanded, Y_manual))


저장된 weight.shape: (2, 3)
weight 객체 id 유지: True
등록된 파라미터: [('weight', (2, 3)), ('bias', (2,))]
수동 계산과 layer 출력 일치: True
b_expanded.shape: (2, 2)
명시적 bias와 broadcasting 일치: True


## 3. 대각 스케일과 대칭 혼합의 순서 바꾸기

이번에는 앞서 손으로 확인한 예제와 다른 숫자를 사용한다.

1. `x @ D @ S`와 `x @ S @ D`의 Shape과 값을 먼저 손으로 예측한다.
2. 두 결과를 계산하고 같은지 확인한다.
3. `D`가 각 성분에 한 일과 `S`가 각 성분에 한 일을 구분한다.
4. 어느 순서에서 어떤 중간 성분이 먼저 확대되었는지 결과 차이와 연결한다.
5. `D`와 `S`가 각각 대칭인지 확인하고, 대칭행렬끼리도 항상 교환되는지 결론을 적는다.


In [11]:
x = torch.tensor([[2.0, -1.0]])
D = torch.diag(torch.tensor([3.0, 0.5]))
S = torch.tensor([[1.0, 1.0], [1.0, -1.0]])

# TODO 3: 순서를 바꾼 두 결과를 계산하세요.
scale_then_mix = x @ D @ S
# scale_then_mix = torch.tensor([5.5, 6.5])
mix_then_scale = x @ S @ D
# mix_then_scale = torch.tensor([3.0, 1.5])

assert scale_then_mix is not None and mix_then_scale is not None

print('D 대칭:', torch.allclose(D, D.T))
print('S 대칭:', torch.allclose(S, S.T))
print('scale_then_mix:', scale_then_mix)
print('mix_then_scale:', mix_then_scale)
print('순서를 바꿔도 같은가:', torch.allclose(scale_then_mix, mix_then_scale))


D 대칭: True
S 대칭: True
scale_then_mix: tensor([[5.5000, 6.5000]])
mix_then_scale: tensor([[3.0000, 1.5000]])
순서를 바꿔도 같은가: False


## 4. 작은 행렬식과 큰 조건수 분리하기

1. 두 행렬 모두 역행렬이 존재하는지 먼저 예측한다.
2. 각 행렬의 `det`와 조건수를 계산한다.
3. 어느 행렬의 `det`가 더 작고 어느 행렬의 조건수가 더 큰지 비교한다.
4. 모든 방향을 같은 비율로 축소하는 경우와 방향별 비율 차이가 큰 경우를 구분해 해석한다.
5. 이 결과만으로 실제 역행렬 계산의 모든 오차를 판단할 수 없는 이유를 한 가지 적는다.


In [12]:
M_uniform = 1e-4 * np.eye(2)
M_uneven = np.diag([1e4, 1e-4])

# TODO 4: 두 행렬의 행렬식과 조건수를 계산하세요.
det_uniform = np.linalg.det(M_uniform)
cond_uniform = np.linalg.cond(M_uniform)
det_uneven = np.linalg.det(M_uneven)
cond_uneven = np.linalg.cond(M_uneven)

assert None not in (det_uniform, cond_uniform, det_uneven, cond_uneven)

print('M_uniform: det =', det_uniform, '/ cond =', cond_uniform)
print('M_uneven : det =', det_uneven, '/ cond =', cond_uneven)
print('uniform 역행렬 존재:', not np.isclose(det_uniform, 0.0, atol=0.0))
print('uneven 역행렬 존재 :', not np.isclose(det_uneven, 0.0, atol=0.0))


M_uniform: det = 1.0000000000000018e-08 / cond = 1.0
M_uneven : det = 1.0000000000000018 / cond = 100000000.0
uniform 역행렬 존재: True
uneven 역행렬 존재 : True


## 막혔을 때 단계별 힌트

<details>
<summary>힌트 1: 행렬곱의 한 칸</summary>

첫 번째 출력은 `X`의 첫 행과 `W_io`의 첫 열을 같은 위치끼리 곱해 더한 뒤 첫 번째 bias를 더한다.
</details>

<details>
<summary>힌트 2: PyTorch weight 배치</summary>

`nn.Linear`의 인자는 입력 성분 수, 출력 성분 수 순서다. 저장된 `weight`는 `(출력, 입력)`이므로 `(입력, 출력)`인 `W_io`와 바로 같은 Shape이 아니다.
</details>

<details>
<summary>힌트 3: 파라미터 값 복사</summary>

새 객체를 대입하기보다 `torch.no_grad()` 안에서 기존 Parameter의 `copy_()`를 사용한다. 복사할 가중치에 전치가 필요한지 Shape부터 비교한다.
</details>

<details>
<summary>힌트 4: bias를 명시적으로 펼치기</summary>

`b`의 앞에 크기 1인 배치 축을 만든 뒤 `Y_manual`과 같은 Shape으로 확장하는 방법을 찾아본다.
</details>

<details>
<summary>힌트 5: 행렬식과 조건수</summary>

NumPy의 `linalg.det`와 `linalg.cond`를 각각 사용한다. 행렬식은 역행렬 존재 여부와 연결하고, 조건수는 방향별 확대·축소 비율의 차이와 연결한다.
</details>


## 결과 해석과 마무리

- 예측한 Shape과 실제 Shape:
- 한 칸의 손계산과 배치 행렬곱이 일치했는지:
- `W_io`가 `layer.weight`에 들어갈 때 전치가 필요했던 이유:
- `copy_()` 전후 객체 `id`와 `named_parameters()`가 보여준 것:
- `bias`를 명시적으로 펼친 결과가 broadcasting에 대해 보여준 것:
- 대각 스케일과 대칭 혼합의 순서를 바꾸자 결과가 어떻게 달라졌으며, 어느 중간 성분 때문에 달라졌는지:
- 두 행렬의 `det`와 조건수 비교가 `det` 크기만으로 안정성을 판단할 수 없다는 점을 어떻게 보여줬는지:
- 이번 작은 고정 행렬 실험의 한계 또는 실패 사례:
